In [1]:
# NLP303 Assessment 3
#
# Detecting AI-generated Text
# Using Transformer-Based Classification
#
#
#
# Tibor Titusz Tarcsai - A00121308
#
# Jonathan Lim - A00142089
#
# Thomas Galindo Salazar - A00129258
#
#
#
# The purpose of this implementation is to demonstrate the building of a working protoype for a Classification task based on the Assessment 2 proposal.
#
#
#***********************
# HOW TO RUN THE CODE:
#***********************
#
# 1. - The notebook can run either in Jupyter Notebook or Google Colab 
#    - If using Google Colab, a Google Drive account is required for data access and storage.
#      Link for running on Colab: 
#
# 2. All helper functions are imported and loaded from the src folder
#
# 3. In Section [2], the "find_cat_dog_lion_tiger_folders" path finding function automatically locates the dataset folder,
#    so NO manual path configuration is required from the user.

In [2]:
# Environment Setup and Dependencies

In [2]:
!pip install -r ../requirements.txt
print("### Dependencies installed successfully ###")

### Dependencies installed successfully ###


In [1]:
# Initial Library Imports
# --------------------------------
# - torch for ... implementation
#
#
#
#
#

import os
import torch
import pandas as pd
import transformers


from torchinfo import summary
from transformers import pipeline

import sys
sys.path.append("../")


from src.downloader import DatasetDownloader

from src.datasetbuilder import DatasetBuilder

from src.preprocessing import TextPreprocessor

from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cuda


# Section 1 - Load Dataset

In [4]:
# Initialises the downloader class
downloader = DatasetDownloader()

# Downloads raw datasets from google drive
print("### Downloading Dataset from Google Drive... ###")
downloader.start_downloading_dataset()

### Downloading Dataset from Google Drive... ###


Downloading...
From: https://drive.google.com/uc?id=16hDeETXMq4-o67YSqguDk6OC3VvXW7U7
To: /home/titus/projects/roberta-ai-text-detector/data/ai_written_humanised_v3_last2500.csv
100%|██████████| 6.47M/6.47M [00:01<00:00, 5.25MB/s]
Downloading...
From: https://drive.google.com/uc?id=1cAL9qHhyYIVnR0kxMBlzRFFyrZqiCf6O
To: /home/titus/projects/roberta-ai-text-detector/data/HUMAN_written_then_AI_polished_v3_last2500.csv
100%|██████████| 8.57M/8.57M [00:01<00:00, 5.54MB/s]
Downloading...
From: https://drive.google.com/uc?id=1LlTrJ6wdRkctvpT21O_XkNTfrHOBhX6L
To: /home/titus/projects/roberta-ai-text-detector/data/ai_generated_v2_last2500.csv
100%|██████████| 5.04M/5.04M [00:02<00:00, 1.82MB/s]
Downloading...
From: https://drive.google.com/uc?id=1Meg3aFojjCGefepim0zVsArmze_5ttGy
To: /home/titus/projects/roberta-ai-text-detector/data/human_written_v2_last2500.csv
100%|██████████| 4.10M/4.10M [00:00<00:00, 5.02MB/s]


In [2]:
# Restructures datasets


raw_dataset_paths = [
            {
                "path": "../data/human_written_v2_last2500.csv"
            },
            {
                "path": "../data/ai_generated_v2_last2500.csv"
            },
            {
                "path": "../data/HUMAN_written_then_AI_polished_v3_last2500.csv"
            },
            {
                "path": "../data/ai_written_humanised_v3_last2500.csv"
            },
        ]


builder = DatasetBuilder()

raw_dataset = builder.build_dataset(
    raw_dataset_paths[0]["path"],
    raw_dataset_paths[1]["path"],
    raw_dataset_paths[2]["path"],
    raw_dataset_paths[3]["path"],
)


In [3]:
print(len(raw_dataset))

10000


In [27]:
raw_dataset.iloc[5049].text

'The middle-levels graph $M_k$ ($0<k\\in\\mathbb{Z}$) has a dihedral quotient pseudograph $R_k$ whose vertices are the $k$-edge ordered trees $T$, each $T$ encoded as a $(2k+1)$-string $F(T)$ formed via $\\rightarrow$DFS by: (i) ($\\leftarrow$BFS-assigned) Kierstead-Trotter lexical colors $0,\\ldots,k$ for the descending nodes; (ii) asterisks $*$ for the $k$ ascending edges. Two ways of corresponding a restricted-growth $k$-string $\\alpha$ to each $T$ exist, namely one Stanley\'s way and a novel way that assigns $F(T)$ to $\\alpha$ via nested substring-swaps. These swaps permit to sort $V(R_k)$ as an ordered tree that allows a lexical visualization of $M_k$ as well as the Hamilton cycles of $M_k$ constructed by P. Gregor, T. M\\"utze and J. Nummenpalo.'

In [ ]:
# Selects rows (delete rows: 5049 )
raw_dataset.iloc[5100:5150]


,text,label,label_name,cleaned_text
5100,The board of Indian conglomerate Reliance has ...,2,human_written_ai_polished,The board of Indian conglomerate Reliance has ...
5101,"In the middle of the night, fifteen-year-old G...",2,human_written_ai_polished,"In the middle of the night, fifteen-year-old G..."
5102,Image segmentation is an important component o...,2,human_written_ai_polished,Image segmentation is an important component o...
5103,European cross-country champion Hayley Yelling...,2,human_written_ai_polished,European cross-country champion Hayley Yelling...
5104,Here Be Dragons (1985) is the first of Penman’...,2,human_written_ai_polished,Here Be Dragons (1985) is the first of Penman’...
5105,"Young book fans have voted Fergus Crane, a sto...",2,human_written_ai_polished,"Young book fans have voted Fergus Crane, a sto..."
5106,"In this novel, Mia must learn to deal with pub...",2,human_written_ai_polished,"In this novel, Mia must learn to deal with pub..."
5107,Semantic segmentation of fine-resolution urban...,2,human_written_ai_polished,Semantic segmentation of fine-resolution urban...
5108,## Hodgson Speaks Out on Winter Break and Cha...,2,human_written_ai_polished,Hodgson Speaks Out on Winter Break and Champi...
5109,The novel was written in 1914. It is set a few...,2,human_written_ai_polished,The novel was written in 1914. It is set a few...


# Section 2 - Text Preprocessing

In [18]:
# Drops rows where 'text' is missing
raw_dataset = raw_dataset.dropna(subset=["text"]).reset_index(drop=True)

print(len(raw_dataset))

9993


In [19]:
text_preprocessor = TextPreprocessor()

raw_dataset["cleaned_text"] = raw_dataset["text"].apply(text_preprocessor.clean_text)



print(raw_dataset[["cleaned_text", "text"]].head(2))

                                        cleaned_text  \
0  For the first time in nearly 3 years i can fin...   
1  Donnie Yen is a long time favorite of mine, al...   

                                                text  
0  For the first time in nearly 3 years i can fin...  
1  Donnie Yen is a long time favorite of mine, al...  


In [ ]:
# give it label 0
df_pure_human = pd.read_csv("../data/human_written_v2_last2500.csv")
df_pure_human.head(2)


,id,adv_source_id,source_id,model,decoding,repetition_penalty,attack,domain,title,prompt,generation
0,0a2c0b33-e74c-4e64-8926-6c79e6170728,0a2c0b33-e74c-4e64-8926-6c79e6170728,0a2c0b33-e74c-4e64-8926-6c79e6170728,human,NaN,NaN,none,reviews,Horrible Bosses,NaN,For the first time in nearly 3 years i can fin...
1,ce4c7569-283a-44fc-83f3-dc06b58c7796,ce4c7569-283a-44fc-83f3-dc06b58c7796,ce4c7569-283a-44fc-83f3-dc06b58c7796,human,NaN,NaN,none,reviews,Ip Man,NaN,"Donnie Yen is a long time favorite of mine, al..."


In [ ]:
# give it label 1
df_pure_ai = pd.read_csv("../data/ai_generated_v2_last2500.csv")
df_pure_ai.head(2)

In [ ]:
df_ai_polished = pd.read_csv("../data/HUMAN_written_then_AI_polished_v3_last2500.csv")
df_ai_polished.head(2)

,id,original_text,rewritten_text,label
0,3811d5e2-3513-49aa-883a-01f847a08c58,GlaxoSmithKline saw its profits fall 9% last y...,GlaxoSmithKline saw its profits fall 9% last y...,2
1,34ee9b7d-d7c9-446f-8c4e-9af9f8bdabf5,"Over the last two decades, deep learning has t...","Over the last two decades, deep learning has t...",2


In [23]:
df_humanised= pd.read_csv("../data/ai_written_humanised_v2.csv")
df_humanised.head(10)

,id,original_text,rewritten_text,label
0,7c168a1f-e4de-4358-b541-1ef7f09651c9,Incorporating Fully Transformed FST into Spars...,This research explores how to improve video-ba...,3
1,d42147ee-420d-481c-91a4-cbef4e789cf8,Abstract: The purpose of this work is to study...,The research presented aims to explore the str...,3
2,525293df-3281-4db8-bd8e-e63a44719b11,An important result in the study of homogeneou...,This study explores the fascinating relationsh...,3
3,5baa1082-0489-4364-9552-e10e7868cede,"In this paper, we present a characterization o...",This paper examines decidable separability for...,3
4,13eabe6e-eb83-461a-909e-fed1b569fdaf,Data compression algorithms have become increa...,My research focuses on studying data compressi...,3
5,ea0add69-e90a-421d-ba19-8217b8c91bfc,The nesting behavior of statistical mechanical...,This study focused on the nested loops problem...,3
6,d69b9054-bef2-409f-bd93-d2c8ff4f6469,An array of antennas could generate very large...,"The use of arrays for directional antennas, wh...",3
7,10749039-0913-449e-b7cd-ae4014af1f53,Adrenaline signaling is an integral part of hu...,The role of adrenaline in human behavior is we...,3
8,8f1a10b7-75a1-43a5-96ee-3f81c6dd8140,This paper investigates the properties of free...,Free ω-continuous and regular ordered algebras...,3
9,a8acd884-8533-479b-98f8-21584986d5a3,Abstract The purpose of this study was to iden...,Examining multiple linear regression datasets ...,3


In [25]:
df_humanised.iloc[8].original_text

'This paper investigates the properties of free ω-continuous and regular ordered algebras. We begin by providing a brief overview of these algebra structures, emphasizing their significance in mathematical logic and semantics. Our main result shows that the class of free ω-continuous orderedalgebras is not closed under taking subdirect products with abelian groups, while theclass of free regularorderedalgebrasisclosedundertakingsubdirectproductswithabeliangroups.Wealsoprovideanexplicitdescriptionofthefreeω-continuousandregularorderedalgebraonanorderedsetofsizeω1.Our findings contribute to the understanding of these algebras and may have implicationsfor other areas of mathematics and computer science.'

In [ ]:
# Section 3 - Tokenisation (BPE)

In [ ]:
# Section 4 - Load Pre-trained RoBERTa

In [ ]:
# Section 5- Fine-tuning

In [ ]:
# Section 6 - Model Inference

In [ ]:
# Section 7 - Threshold Classification

In [ ]:
# Section 8 - Evaluation

In [ ]:
# Section 9 - Visualisations

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

model = AutoModelForSequenceClassification.from_pretrained(
    "fakespot-ai/roberta-base-ai-text-detection-v1"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
# Prints model summary
summary(model, input_size=(1, 512), dtypes=[torch.long])

Layer (type:depth-idx)                                       Output Shape              Param #
RobertaForSequenceClassification                             [1, 2]                    --
├─RobertaModel: 1-1                                          [1, 512, 768]             --
│    └─RobertaEmbeddings: 2-1                                [1, 512, 768]             --
│    │    └─Embedding: 3-1                                   [1, 512, 768]             38,603,520
│    │    └─Embedding: 3-2                                   [1, 512, 768]             768
│    │    └─Embedding: 3-3                                   [1, 512, 768]             394,752
│    │    └─LayerNorm: 3-4                                   [1, 512, 768]             1,536
│    │    └─Dropout: 3-5                                     [1, 512, 768]             --
│    └─RobertaEncoder: 2-2                                   [1, 512, 768]             --
│    │    └─ModuleList: 3-6                                  --               

In [ ]:
df = pd.read_csv("data/HUMAN_written_then_AI_polished.csv")

In [ ]:
print(df.head(200))

                                      id  \
0   e5e058ce-be2b-459d-af36-32532aaba5ff   
1   f95b107b-d176-4af5-90f7-4d0bb20caf93   
2   856d8972-9e3d-4544-babc-0fe16f21e04d   
3   fbc8a5ea-90fa-47b8-8fa7-73dd954f1524   
4   72c41b8d-0069-4886-b734-a4000ffca286   
5   72fe360b-cce6-4daf-b66a-1d778f5964f8   
6   df594cf4-9a0c-4488-bcb3-68f41e2d5a16   
7   853c0e51-7dd5-4bb5-8286-e4aa8820173b   
8   1649f195-8f98-4c79-92b6-54a5ca9261fa   
9   5e23ab14-b85f-48e8-9aa3-15452e73524e   
10  ddcb207c-a790-4e16-a053-4aced58d7c15   
11  b00bf7dc-4de9-4ab4-9962-a16e0b5f4628   
12  04d3809c-0abe-4bee-b1d2-9787af95362f   
13  06bffeb2-bea0-4b0b-b60d-767ba9b660a7   
14  5eb88a59-eb5a-49ea-8304-f67efe338921   
15  1389aa64-25fb-4e56-9358-ef34143bfea9   
16  d0064195-c22e-4550-a265-6b372deea3e0   
17  417afaa2-2d21-4df1-953b-768647de9980   
18  ce898c28-428f-446f-975e-a1265942f2da   
19  380cd71d-3300-422c-9cde-8a63e71f2797   
20  c093400c-2bd2-4e0d-a732-f99d499d58a9   
21  05f40b6d-67cf-4a6e-ad2f-cfe0